# Gaussian Splats

In [1]:
plyPath = r"C:\Users\jelle\Documents\DoctoraatLocal\generationtools\data\CompletionChairSplat.ply"

In [2]:
import trimesh

splat = trimesh.load(plyPath)
splat.show()

In [2]:
from plyfile import PlyData, PlyElement
import numpy as np
import open3d as o3d

ply = PlyData.read(plyPath)
vertex = ply['vertex'].data

# Convert to a dict of numpy arrays
splat = {name: np.asarray(vertex[name]) for name in vertex.dtype.names}

print("Available fields:", splat.keys())
print("Example x positions:", splat['x'][:5])

positions = np.stack([splat['x'], splat['y'], splat['z']], axis=1)
scales = np.stack([splat['scale_0'], splat['scale_1'], splat['scale_2']], axis=1)
colors = np.stack([splat['f_dc_0'],
                  splat['f_dc_1'],
                  splat['f_dc_2']], axis=1)
# Create Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(positions)

if colors is not None:
    pcd.colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([pcd])



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Available fields: dict_keys(['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3'])
Example x positions: [-0.2952162  -0.29864824 -0.29644734 -0.2970978  -0.29640317]


## Editing Splats

In [5]:
from context import generationtools as gnt

#Set the positions and axis for the planes
startEndRange = np.array([0.35,0.65])  # Positions along the chosen axis normalised from 0 to 1
axis = 0  # Axis for the planes ('x', 'y', or 'z')
bbSize = 1

# Create the planes
planes = [gnt.create_transparent_plane(pos, axis=axis, size = bbSize, backend = "o3d") for pos in startEndRange]

# Create a scene and add the mesh and planes
planes.append(pcd)
o3d.visualization.draw_geometries(planes)

In [12]:
scaleFactor = 2

# Calculate the new distances
minVal = -bbSize/2.0 + startEndRange[0] * bbSize
maxVal = -bbSize/2.0 + startEndRange[1] * bbSize
distance = np.abs(maxVal-minVal)
newDist = distance * scaleFactor

points = np.asarray(pcd.points)

for point in points:
        # Move the plane to the specified axis
    if(point[axis] > minVal):
        # The point is further than the startplane
        if(point[axis] > maxVal):
            # The point is further than the end plane -> just move the max distance
            point[axis] += newDist - distance
        else:
            # The point is in between -> interpolate
            t = (point[axis] - minVal)/distance
            point[axis] += newDist * t/2

pcd.points =  o3d.utility.Vector3dVector(points)
o3d.visualization.draw_geometries([pcd])



In [13]:
# Write back to the ply
splat['x'] = points[:, 0]
splat['y'] = points[:, 1]
splat['z'] = points[:, 2]

# Reconstruct structured array
vertex_dtype = vertex.dtype
vertex_array = np.empty(len(splat['x']), dtype=vertex_dtype)

for name in vertex_dtype.names:
    vertex_array[name] = splat[name]

new_vertex = PlyElement.describe(vertex_array, 'vertex')

new_ply = PlyData([new_vertex], text=ply.text)
new_ply.write("scaled_output.ply")

## Scaling splats

In [25]:
import math
scaleFactor = 5

# Calculate the new distances
minVal = -bbSize/2.0 + startEndRange[0] * bbSize
maxVal = -bbSize/2.0 + startEndRange[1] * bbSize
distance = np.abs(maxVal-minVal)
newDist = distance * scaleFactor

newPositions = np.copy(positions)
newScales = np.copy(scales)
i = 0
for point in newPositions:
        # Move the plane to the specified axis
    if(point[axis] > minVal):
        # The point is further than the startplane
        if(point[axis] > maxVal):
            # The point is further than the end plane -> just move the max distance
            point[axis] += newDist - distance
        else:
            # The point is in between -> interpolate
            oldDist = point[axis]-minVal
            point[axis] = minVal + oldDist *scaleFactor 
            
            # scale local axes proportionally
            newScales[i] /= math.sqrt(math.sqrt(math.sqrt(scaleFactor)))
    i+=1

In [26]:
# Write back to the ply
splat['x'] = newPositions[:, 0]
splat['y'] = newPositions[:, 1]
splat['z'] = newPositions[:, 2]
splat['scale_0'] = newScales[:, 0]
splat['scale_1'] = newScales[:, 1]
splat['scale_2'] = newScales[:, 2]

# Reconstruct structured array
vertex_dtype = vertex.dtype
vertex_array = np.empty(len(splat['x']), dtype=vertex_dtype)

for name in vertex_dtype.names:
    vertex_array[name] = splat[name]

new_vertex = PlyElement.describe(vertex_array, 'vertex')

new_ply = PlyData([new_vertex], text=ply.text)
new_ply.write("scaled_output.ply")